# 184. Department Highest Salary

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** database, join, group-by, window-function
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/department-highest-salary/)

```
Table: Employee                        Table: Department
+--------------+---------+             +-------------+---------+
| Column Name  | Type    |             | Column Name | Type    |
+--------------+---------+             +-------------+---------+
| id           | int     |             | id          | int     |
| name         | varchar |             | name        | varchar |
| salary       | int     |             +-------------+---------+
| departmentId | int     |             id is the primary key.
+--------------+---------+
id is the primary key. departmentId is a foreign key to Department.
```

Write a solution to find employees who have the **highest salary** in each of the
departments.

Return the result table in **any order**. The result columns must be called
`Department`, `Employee` and `Salary`.

---

### Example

```
Employee:                                   Department:
+----+-------+--------+--------------+      +----+-------+
| id | name  | salary | departmentId |      | id | name  |
+----+-------+--------+--------------+      +----+-------+
| 1  | Joe   | 70000  | 1            |      | 1  | IT    |
| 2  | Jim   | 90000  | 1            |      | 2  | Sales |
| 3  | Henry | 80000  | 2            |      +----+-------+
| 4  | Sam   | 60000  | 2            |
| 5  | Max   | 90000  | 1            |
+----+-------+--------+--------------+

Output:
+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Jim      | 90000  |
| Sales      | Henry    | 80000  |
| IT         | Max      | 90000  |
+------------+----------+--------+
```

Both Jim **and** Max earn 90000 in IT, so **both** are reported.

---

The first problem here that needs a group **and** the rows the group came from - and
`GROUP BY` destroys exactly the information you need. Look at the output again: two IT
rows. Any answer that reports one row per department is wrong before it starts.

## Before you write anything

**1.** Run `SELECT departmentId, MAX(salary) FROM Employee GROUP BY departmentId` with
`show`. You now have the right salaries and **no names**. Say precisely why the name is
not there and cannot be - two employees are in the group and the group has one row, so
which name would it be? (This is #182's question 1, and it is the reason this problem is
harder than it looks.)

**2.** So the shape is: work out the maximum per department **first**, then go back to
`Employee` and pull out the rows that match it. Write that as an `IN` over a **pair**:

```sql
WHERE (departmentId, salary) IN (SELECT departmentId, MAX(salary) FROM Employee GROUP BY departmentId)
```

Row-value comparison like this works in SQLite and MySQL. Say why matching on `salary`
**alone** would be wrong, and construct the two-department dataset that proves it.

**3.** **Ties.** Two people earn 90000 in IT and both must appear. Check that your
approach returns both rather than picking one - and note that this is the requirement
that rules out "join to the max and take one row".

**4.** You need the department **name**, which lives in the other table. Which join, and
which direction? Does a department with **no employees** appear in the output? Should it?
(Read the expected output again - there is a test for it.)

**5.** The window-function version: `MAX(salary) OVER (PARTITION BY departmentId)` gives
every row its own department's maximum, **without collapsing the rows**. Write it, then
say in one sentence what `PARTITION BY` does differently from `GROUP BY`. That sentence
is the single most useful thing in this notebook.

**6.** Output columns are `Department`, `Employee`, `Salary` - none of which is what the
columns are called in the tables. Write the three `AS` clauses, and note that `name`
appears in **both** tables, so you must qualify it.

## Two routes

**A - group first, then match back** *(write this first)*

```sql
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON d.id = e.departmentId
WHERE (e.departmentId, e.salary) IN (
    SELECT departmentId, MAX(salary) FROM Employee GROUP BY departmentId
)
```

The subquery gives one `(department, top salary)` pair per department; the outer query
returns **every** employee row matching one of those pairs, which is what makes ties
work automatically. The inner `JOIN` (not `LEFT JOIN`) is right here: a department with
no employees has no maximum and should not appear.

Matching on the pair rather than on `salary` alone is question 2's point - otherwise
someone earning 90000 in Sales would be reported because 90000 happens to be IT's
maximum.

**B - `PARTITION BY`, the window version**

```sql
SELECT Department, Employee, Salary FROM (
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
           MAX(e.salary) OVER (PARTITION BY e.departmentId) AS top
    FROM Employee e
    JOIN Department d ON d.id = e.departmentId
) t
WHERE Salary = top
```

`PARTITION BY` splits the rows into groups **and leaves every row in place**, attaching
the group's `MAX` to each one. `GROUP BY` collapses; `PARTITION BY` annotates. That is
question 5's sentence, and once it lands, a whole category of "the group *and* the rows"
problems becomes routine - including **#185**, which is this query with `DENSE_RANK`
instead of `MAX`.

> **`GROUP BY` collapses rows; `PARTITION BY` keeps them.** Every problem that asks for
> "the rows that hit the group's maximum" is really asking you to compute a per-group
> value without losing the rows, and those are the two ways to do it. Route A is the
> classic; route B is the one that scales to top-3, top-N, running totals and
> percentages of the group.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER);
CREATE TABLE Department (id INTEGER, name TEXT);"""

EXPECTED_COLUMNS = ['Department', 'Employee', 'Salary']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Employee VALUES (1,'Joe',70000,1),(2,'Jim',90000,1),(3,'Henry',80000,2),
                            (4,'Sam',60000,2),(5,'Max',90000,1);
INSERT INTO Department VALUES (1,'IT'),(2,'Sales');
''', [('IT','Jim',90000), ('Sales','Henry',80000), ('IT','Max',90000)])

check("question 3: a three-way tie - all three are reported", '''
INSERT INTO Employee VALUES (1,'A',500,1),(2,'B',500,1),(3,'C',500,1),(4,'D',100,1);
INSERT INTO Department VALUES (1,'Eng');
''', [('Eng','A',500), ('Eng','B',500), ('Eng','C',500)])

check("one department, one employee", '''
INSERT INTO Employee VALUES (1,'Solo',1,1);
INSERT INTO Department VALUES (1,'Only');
''', [('Only','Solo',1)])

check("question 4: a department with NO employees does not appear", '''
INSERT INTO Employee VALUES (1,'A',10,1);
INSERT INTO Department VALUES (1,'Busy'),(2,'Empty');
''', [('Busy','A',10)])

check("*** question 2: the same salary is top in one dept and not in another ***", '''
INSERT INTO Employee VALUES (1,'Top',90000,1),(2,'Mid',90000,2),(3,'Boss',95000,2);
INSERT INTO Department VALUES (1,'IT'),(2,'Sales');
''', [('IT','Top',90000), ('Sales','Boss',95000)])

check("many departments", '''
INSERT INTO Employee VALUES (1,'A',10,1),(2,'B',20,2),(3,'C',30,3),
                            (4,'D',5,1),(5,'E',25,2),(6,'F',35,3);
INSERT INTO Department VALUES (1,'One'),(2,'Two'),(3,'Three');
''', [('One','A',10), ('Two','E',25), ('Three','F',35)])

check("everyone in one department earns the same", '''
INSERT INTO Employee VALUES (1,'A',7,1),(2,'B',7,1);
INSERT INTO Department VALUES (1,'Flat');
''', [('Flat','A',7), ('Flat','B',7)])

check("negative and zero salaries", '''
INSERT INTO Employee VALUES (1,'A',0,1),(2,'B',-5,1);
INSERT INTO Department VALUES (1,'Odd');
''', [('Odd','A',0)])

check("no employees at all", '''
INSERT INTO Department VALUES (1,'Ghost');
''', [])

check("both tables empty", '', [])

## After it passes

- **Prove question 2 the hard way.** Change your subquery to `WHERE e.salary IN (SELECT
  MAX(salary) FROM Employee GROUP BY departmentId)` - dropping the department from the
  match - and run the tests. The case named for it fails, and the extra row it returns
  is somebody who is not the top earner anywhere. That is the whole reason for matching
  on the pair.
- **Break the tie handling.** Try `JOIN (SELECT departmentId, MAX(salary) ...) ` and then
  `GROUP BY departmentId` on the outside. You will get one IT row instead of two. Ties
  are the requirement that most "obvious" answers quietly fail.
- **Write out question 5's sentence** and keep it: `GROUP BY` collapses, `PARTITION BY`
  annotates. Then read route B again and see that the whole query is one `WHERE` on top
  of an annotation.
- **Then go straight to #185**, which is this problem with `DENSE_RANK() <= 3` where you
  currently have `MAX`. If route B made sense, #185 is twenty minutes; if it did not,
  #185 will be painful. That is a good way to find out whether it landed.
- **Make it real.** Add the *number* of employees in each department and each top
  earner's share of the department payroll. Both are one more window function over the
  same partition - `COUNT(*) OVER (...)` and `SUM(salary) OVER (...)` - and route A
  cannot do either without a second subquery.
- Siblings: **#185 Department Top Three Salaries** (do this next), #176 Second Highest
  Salary (the same "rank within a set" idea without groups), #178 Rank Scores,
  #1112 Highest Grade For Each Student.